# Codi homografia i tractament de les dades 3D

# Processar imatge .ply

In [ ]:
# Instal·la les llibreries si fa falta:
# !pip install open3d opencv-python numpy matplotlib

import open3d as o3d
import numpy as np
import cv2
import matplotlib.pyplot as plt

def ply_a_mapa_profunditat(ruta_ply, resolucio=(640, 640)):
    """
    Llegeix el .ply i el passa a mapa de profunditat 2D.
    """
    print(f"Llegint {ruta_ply}...")

    # Carregar el núvol de punts
    pcd = o3d.io.read_point_cloud(ruta_ply)

    # Comprovar que s'ha carregat bé
    if not pcd.has_points():
        raise ValueError("No s'han pogut llegir els punts del PLY.")

    # Guardar coordenades
    punts = np.asarray(pcd.points)

    # Passar a 2D
    # Ens quedem X i Y, Z és la profunditat
    x = punts[:, 0]
    y = punts[:, 1]
    z = punts[:, 2]

    # Normalitzar X i Y per la resolució
    # Ajustar entre 0 i 1
    x_norm = (x - np.min(x)) / (np.max(x) - np.min(x))
    y_norm = (y - np.min(y)) / (np.max(y) - np.min(y))

    # Passar a píxels
    x_pixels = (x_norm * (resolucio[0] - 1)).astype(int)
    y_pixels = (y_norm * (resolucio[1] - 1)).astype(int)

    # Crear imatge de profunditat
    # Matriu buida
    imatge_profunditat = np.zeros((resolucio[1], resolucio[0]), dtype=np.float32)

    # Assignar Z als píxels
    # Si coincideixen diversos punts, ens quedem el més alt
    imatge_profunditat.fill(-np.inf)

    # Gestionar col·lisions de punts
    np.maximum.at(imatge_profunditat, (y_pixels, x_pixels), z)

    # Omplir forats
    # Buscar píxels buits
    mascara_invalida = np.isinf(imatge_profunditat)

    # Donar valor de fons als forats
    min_z_valid = np.min(z)
    imatge_profunditat[mascara_invalida] = min_z_valid

    # Passar a 8 bits per poder usar OpenCV
    max_z = np.max(imatge_profunditat)
    min_z = np.min(imatge_profunditat)

    imatge_8bits = 255 * (imatge_profunditat - min_z) / (max_z - min_z)
    imatge_8bits = imatge_8bits.astype(np.uint8)

    # Suavitzar una mica la imatge
    imatge_8bits = cv2.medianBlur(imatge_8bits, 5)

    return imatge_8bits

# Provar el codi

# Ruta del .ply
# ruta_arxiu_ply = "/content/porcs_granja1_001.ply"
ruta_arxiu_ply = "3D_images\blaze-112-40676064-0008-points_intensity.ply"

try:
    # Convertir a 2D
    mapa_2d = ply_a_mapa_profunditat(ruta_arxiu_ply)

    # Guardar la imatge
    cv2.imwrite("mapa_profunditat_sortida.png", mapa_2d)
    print("Imatge guardada com a mapa_profunditat_sortida.png")

except Exception as e:
    print(f"Error: {e}")
    print("Revisa la ruta del fitxer.")


# Homografia

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

# Configurar les rutes
ruta_video = os.path.join('Video_definitiu_homografia', 'room2_cam6_2026-04-15_20-15-05.ts')
ruta_img_ply = 'mapa_profunditat_sortida.png'

# Comprovar si tenim els fitxers
if not os.path.exists(ruta_video):
    # Buscar-lo a l'arrel si no està a la carpeta
    ruta_video_alt = 'room2_cam6_2026-04-15_20-15-05.ts'
    if os.path.exists(ruta_video_alt):
        ruta_video = ruta_video_alt
    else:
        print(f"No trobo el vídeo a {ruta_video}")
        exit()

if not os.path.exists(ruta_img_ply):
    print(f"No trobo la imatge 3D a {ruta_img_ply}")
    exit()

print(f"Tenim el vídeo a: {ruta_video}")
print(f"Tenim el mapa 3D a: {ruta_img_ply}")

# Triar frames del vídeo
cap = cv2.VideoCapture(ruta_video)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv2.CAP_PROP_FPS)

print(f"El vídeo té {total_frames} frames.")
print("Vols triar els frames a mà (1) o que ho faci automàtic (2)?")

opcio = input("Escriu 1 o 2: ").strip()

frames_seleccionats = []

if opcio == '1':
    print("Navega pel vídeo:")
    print(" - 'a' / 'd' per moure't 10 frames.")
    print(" - 's' / 'w' per moure't 100 frames.")
    print(" - 'c' per guardar el frame.")
    print(" - 'q' per sortir.")

    cv2.namedWindow("Vídeo", cv2.WINDOW_NORMAL)
    cv2.resizeWindow("Vídeo", 960, 540)

    current_frame_idx = 0

    def on_trackbar(val):
        global current_frame_idx
        current_frame_idx = val

    cv2.createTrackbar("Frame", "Vídeo", 0, total_frames - 1, on_trackbar)

    while len(frames_seleccionats) < 3:
        cap.set(cv2.CAP_PROP_POS_FRAMES, current_frame_idx)
        ret, frame = cap.read()
        if not ret:
            current_frame_idx = max(0, current_frame_idx - 1)
            continue
        frame = cv2.flip(frame, 1)

        frame_visual = frame.copy()
        cv2.putText(frame_visual, f"Frame: {current_frame_idx} | Triats: {len(frames_seleccionats)}/3", (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2)

        cv2.imshow("Vídeo", frame_visual)

        tecla = cv2.waitKey(30) & 0xFF

        if tecla == ord('q'):
            cv2.destroyAllWindows()
            break
        elif tecla == ord('c'):
            frames_seleccionats.append((current_frame_idx, frame.copy()))
            print(f"M'he guardat el frame {current_frame_idx}")
        elif tecla == ord('d'):
            current_frame_idx = min(total_frames - 1, current_frame_idx + 10)
            cv2.setTrackbarPos("Frame", "Vídeo", current_frame_idx)
        elif tecla == ord('a'):
            current_frame_idx = max(0, current_frame_idx - 10)
            cv2.setTrackbarPos("Frame", "Vídeo", current_frame_idx)
        elif tecla == ord('w'):
            current_frame_idx = min(total_frames - 1, current_frame_idx + 100)
            cv2.setTrackbarPos("Frame", "Vídeo", current_frame_idx)
        elif tecla == ord('s'):
            current_frame_idx = max(0, current_frame_idx - 100)
            cv2.setTrackbarPos("Frame", "Vídeo", current_frame_idx)

    cv2.destroyAllWindows()

# Extracció automàtica si no s'han triat
if len(frames_seleccionats) < 3:
    print("Vaig a agafar-los automàticament.")
    punts_de_tall = [int(total_frames * 0.25), int(total_frames * 0.50), int(total_frames * 0.75)]
    frames_seleccionats = []

    for idx in punts_de_tall:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            frame = cv2.flip(frame, 1)
            frames_seleccionats.append((idx, frame))

cap.release()

if len(frames_seleccionats) < 3:
    print("No he pogut agafar els frames.")
    exit()

# Guardar frames escollits
for i, (idx, img) in enumerate(frames_seleccionats):
    cv2.imwrite(f"frame_{i+1}.jpg", img)

# Retallar imatge 3D
print("Ara retallarem la imatge 3D.")
img_ply_full = cv2.imread(ruta_img_ply)
img_ply_full = cv2.cvtColor(img_ply_full, cv2.COLOR_BGR2RGB)
img_ply_full = cv2.rotate(img_ply_full, cv2.ROTATE_90_COUNTERCLOCKWISE)

print("Selecciona el tros que vols amb el ratolí i prem ENTER.")
img_ply_bgr = cv2.cvtColor(img_ply_full, cv2.COLOR_RGB2BGR)
r = cv2.selectROI("Retall", img_ply_bgr, showCrosshair=True, fromCenter=False)
cv2.destroyWindow("Retall")

if r == (0, 0, 0, 0):
    print("No has retallat, ho agafo tot.")
    img_ply = img_ply_full.copy()
else:
    img_ply = img_ply_full[int(r[1]):int(r[1]+r[3]), int(r[0]):int(r[0]+r[2])]
    print("He retallat la imatge.")

# Escollir punts
def seleccionar_punts(imatge, titol):
    plt.figure(figsize=(10, 8))
    plt.imshow(imatge)
    plt.title(f"{titol}\nClica 4 punts de referència.")
    plt.axis('off')
    punts = plt.ginput(4, timeout=0)
    plt.close()
    return np.array(punts, dtype=np.float32)

print("Clica 4 punts clars a la imatge 3D.")
punts_ply = seleccionar_punts(img_ply, "Càmera 3D")

if len(punts_ply) < 4:
    print("Et falten punts.")
    exit()

matrius_calculades = []

# Calcular homografies pels frames
for i, (idx, frame_bgr) in enumerate(frames_seleccionats):
    print(f"Processant frame {i+1}...")
    img_2d = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

    print("Clica els mateixos 4 punts en aquesta imatge.")
    punts_2d = seleccionar_punts(img_2d, f"Frame {i+1}")

    if len(punts_2d) < 4:
        print("No hi ha prou punts, salto aquest.")
        continue

    # Calcular matriu
    H, mask = cv2.findHomography(punts_2d, punts_ply, cv2.RANSAC, 5.0)
    matrius_calculades.append((idx, H))

    # Fer un polígon de prova
    alt, ampl = img_2d.shape[:2]
    startX, startY = int(ampl * 0.45), int(alt * 0.45)
    width, height = int(ampl * 0.12), int(alt * 0.18)

    poligon_2d = np.array([
        [startX, startY],
        [startX + width, startY],
        [startX + width, startY + height],
        [startX, startY + height]
    ], dtype=np.float32).reshape(-1, 1, 2)

    poligon_projectat = cv2.perspectiveTransform(poligon_2d, H)

    img_2d_dibux = img_2d.copy()
    img_ply_dibux = img_ply.copy()

    # Pintar punts i polígons
    for pt in punts_2d:
        cv2.circle(img_2d_dibux, (int(pt[0]), int(pt[1])), 8, (255, 0, 0), -1)
    for pt in punts_ply:
        cv2.circle(img_ply_dibux, (int(pt[0]), int(pt[1])), 8, (255, 0, 0), -1)

    cv2.polylines(img_2d_dibux, [np.int32(poligon_2d)], True, (0, 255, 0), 3)
    cv2.polylines(img_ply_dibux, [np.int32(poligon_projectat)], True, (0, 255, 0), 3)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))
    plt.subplots_adjust(left=0, right=1, top=1, bottom=0, wspace=0.02, hspace=0)

    ax1.imshow(img_2d_dibux)
    ax1.axis('off')
    ax2.imshow(img_ply_dibux)
    ax2.axis('off')

    fig.savefig(f"resultat_frame_{i+1}.jpg", dpi=300, bbox_inches='tight', pad_inches=0)
    plt.close(fig)

# Guardar matrius en un .txt
ruta_matrius = 'matrius_homografia.txt'
with open(ruta_matrius, 'w', encoding='utf-8') as f:
    f.write("Matrius calculades:\n\n")
    for idx, (f_idx, H) in enumerate(matrius_calculades):
        f.write(f"Frame {idx+1}:\n")
        np.savetxt(f, H, fmt='%0.8f')
        f.write("\n")

print("Tot guardat! Hem acabat.")
